## Viva points

- **K-Means:** pick k centres, assign each point to the nearest centre, move centres to the mean, repeat.
- **Elbow method:** plot inertia against k and pick the bend.
- **Hierarchical (agglomerative):** start with every point as its own cluster and keep merging the two closest clusters.
- **Dendrogram:** a tree of the merges. Cutting it at a height gives the clusters.
- Scaling is important because both use distances.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

# 1. Load real-world Iris dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)

# Scale the features
X_scaled = StandardScaler().fit_transform(X)

# 2. Elbow Method
wcss = []

for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    wcss.append(model.inertia_)

plt.plot(range(1, 11), wcss, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

# 3. Dendrogram
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(10, 5))
dendrogram(Z, truncate_mode="lastp", p=20)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Clusters")
plt.ylabel("Distance")
plt.show()

# Choose K based on the Elbow Method and Dendrogram
k = 3

# 4. K-Means Clustering
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

# 5. Hierarchical Clustering
hierarchical = AgglomerativeClustering(n_clusters=k)
hierarchical_labels = hierarchical.fit_predict(X_scaled)

# 6. Analyze the clusters
X["KMeans_Cluster"] = kmeans_labels
X["Hierarchical_Cluster"] = hierarchical_labels

print("K-Means Cluster Counts:")
print(X["KMeans_Cluster"].value_counts().sort_index())

print("\nHierarchical Cluster Counts:")
print(X["Hierarchical_Cluster"].value_counts().sort_index())

print("\nK-Means Silhouette Score:",
      round(silhouette_score(X_scaled, kmeans_labels), 3))

print("Hierarchical Silhouette Score:",
      round(silhouette_score(X_scaled, hierarchical_labels), 3))

# 7. Visualize K-Means clusters
plt.scatter(X_scaled[:, 0], X_scaled[:, 1],
            c=kmeans_labels, cmap="viridis")
plt.xlabel("Sepal Length (scaled)")
plt.ylabel("Sepal Width (scaled)")
plt.title("K-Means Clustering")
plt.show()
